# **Thông tin nhóm**
- Lớp: ML Labs 23KHDL1
- Nhóm: 4
- Sinh viên: 
    - 23127102 - Lê Quang Phúc
    - 23127212 - Nguyễn Quang Đăng Khoa
    - 23127241 - Đoàn Thành Phát
    - 23127332 - Trần Tiến Cường
    - 23127442 - Trầm Hữu Nhân


# **Tiền xử lý dữ liệu**

## 1. Thư viện

In [ ]:
import os
import glob
import pandas as pd
import re
import ast
import unicodedata 
import time
import json

from dotenv import load_dotenv
from google import genai
from tqdm import tqdm
from google.genai import types
from pydantic import BaseModel
from urllib.parse import urlparse, urlencode, parse_qs

## 2. Tổng hợp dữ liệu 

In [36]:
import os
import glob
import pandas as pd

data_dir = os.path.join(os.path.dirname(os.getcwd()), 'data')
output_file = os.path.join(data_dir, 'data.csv')
csv_files = sorted(glob.glob(os.path.join(data_dir, '*.csv')))

header = ['name', 'address', 'score', 'nums_comments', 'price', 'category', 'hours', 'comments', 'comment_scores', 'url']

dfs = []

for csv_file in csv_files:
    if os.path.basename(csv_file) == 'data.csv':
        continue
        
    try:
        df = pd.read_csv(csv_file)
        df = df.iloc[:, :10]
        df.columns = header
        df = df.dropna(how='all')
        dfs.append(df)
    except Exception as e:
        print(f"Bỏ qua file {os.path.basename(csv_file)} vì lỗi: {e}")

if dfs:
    combined_df = pd.concat(dfs, ignore_index=True)
    combined_df.to_csv(output_file, index=False, encoding='utf-8-sig')

data = pd.read_csv(output_file)
print(f"Kích thước của bộ dữ liệu (dòng, cột): {data.shape}")

Kích thước của bộ dữ liệu (dòng, cột): (6050, 10)


In [45]:
# Read csv file
data = pd.read_csv('../data/data.csv')

## 3. Tổng quan về dữ liệu

### 3.1 Ngữ cảnh
Đây là bộ dữ liệu tập hợp các thông tin được thu thập từ các nền tảng (Google Maps, Booking, Foody) tập trung chủ yếu vào các dịch vụ lưu trú, nhà hàng, quán ăn và các địa điểm du lịch ở Thành phố Hồ Chí Minh. Để có thể tiện sử dụng cho các giai đoạn sau này, nhóm sẽ tập hợp dữ liệu lại thành một file duy nhất `data.csv`.

### 3.2 Mô tả đặc trưng


In [46]:
# 1. Định nghĩa từ điển mô tả cho các cột trong data.csv
column_desc = {
    'name': ['Tên địa điểm', '-'],
    'address': ['Địa chỉ', '-'],
    'score': ['Điểm đánh giá', 'Sao (0.0 - 5.0)'],
    'nums_comments': ['Số lượng bình luận', 'Bình luận'],
    'price': ['Giá dịch vụ', 'VNĐ'],
    'category': ['Nhãn dữ liệu', 'Attraction, Hotel, Dinning'],
    'hours': ['Giờ hoạt động', 'Giờ mở cửa - Giờ đóng cửa'],
    'comments': ['Bình luận đánh giá', 'Danh sách chuỗi'],
    'comment_scores': ['Điểm đánh giá của bình luận', 'Danh sách số'],
    'url': ['Đường dẫn', '-']
}

# 2. Tạo DataFrame từ kiểu dữ liệu (dtypes) của bộ dữ liệu
data_dict = pd.DataFrame(data.dtypes, columns=['Kiểu dữ liệu'])
data_dict.reset_index(inplace=True)
data_dict.rename(columns={'index': 'Thuộc tính'}, inplace=True)

# 3. Ánh xạ (map) Description và Unit từ dictionary vào DataFrame
data_dict['Mô tả'] = data_dict['Thuộc tính'].apply(lambda x: column_desc.get(x, ['N/A', 'N/A'])[0])
data_dict['Đơn vị'] = data_dict['Thuộc tính'].apply(lambda x: column_desc.get(x, ['N/A', 'N/A'])[1])

# 4. Sắp xếp lại thứ tự các cột cho dễ nhìn
data_dict = data_dict[['Thuộc tính', 'Mô tả', 'Đơn vị', 'Kiểu dữ liệu']]

# 5. Hiển thị bảng Data Dictionary với định dạng căn trái
styled_dict = data_dict.style.hide(axis='index') \
    .set_properties(**{'text-align': 'left'}) \
    .set_table_styles([{'selector': 'th', 'props': [('text-align', 'left')]}])

display(styled_dict)

Thuộc tính,Mô tả,Đơn vị,Kiểu dữ liệu
name,Tên địa điểm,-,str
address,Địa chỉ,-,str
score,Điểm đánh giá,Sao (0.0 - 5.0),float64
nums_comments,Số lượng bình luận,Bình luận,float64
price,Giá dịch vụ,VNĐ,str
category,Nhãn dữ liệu,"Attraction, Hotel, Dinning",str
hours,Giờ hoạt động,Giờ mở cửa - Giờ đóng cửa,str
comments,Bình luận đánh giá,Danh sách chuỗi,str
comment_scores,Điểm đánh giá của bình luận,Danh sách số,str
url,Đường dẫn,-,str


### 3.3 Thông tin dữ liệu

In [39]:
print(f"- Kích thước của bộ dữ liệu (dòng, cột): {data.shape}")

# Những dòng dầu
display(data.head())
print("\n")

# Thông tin tổng quan về bộ dữ liệu
data.info()
print("\n")

- Kích thước của bộ dữ liệu (dòng, cột): (6050, 10)


,name,address,score,nums_comments,price,category,hours,comments,comment_scores,url
0,*BOM HOMES* VINHOMES CENTRAL PARK- LUXURY APAR...,"208 Nguyễn Hữu Cảnh, Quận Bình Thạnh, TP. Hồ ...",0.5,3.0,1350000,hotel,00:00 - 23:59,['Quá tệ. Booking xác nhận. Chủ nhà hủy ko báo...,"[0,5; 0,5; 0,5]",https://www.booking.com/hotel/vn/bom-homes-vin...
1,1 AM Home Sai Gon,109/14 Đường Nguyễn Thiện Thuật Toà nhà có tần...,4.5,8.0,770000,hotel,00:00 - 23:59,"['Phòng đẹp ơi là đẹp, rộng rãi thoải mái, sán...",[5; 5; 5; 4; 4; 4; 5; 4],https://www.booking.com/hotel/vn/1-am-homestay...
2,1 Bedroom Apartment Cozy Canal View Hoang Sa s...,"287 Đường Hoàng Sa, Quận 1, 123080 TP. Hồ Chi...",0.0,0.0,1399000,hotel,00:00 - 23:59,[],[],https://www.booking.com/hotel/vn/1-bedroom-apa...
3,1 Bedroom Apartment Landmark Plus Lp,"208 Đường Nguyễn Hữu Cảnh, Quận Bình Thạnh, TP...",4.5,1.0,1288888,hotel,00:00 - 23:59,['The support and friendliness from the staff....,"[4,5]",https://www.booking.com/hotel/vn/1-bedroom-apa...
4,1 Bedroom Daisy - Luxury Gold Apartment - Cent...,"538/2/5 Đoàn Văn Bơ, 72800 TP. Hồ Chí Minh, ...",0.0,0.0,750000,hotel,00:00 - 23:59,[],[],https://www.booking.com/hotel/vn/1-bedroom-dai...




<class 'pandas.DataFrame'>
RangeIndex: 6050 entries, 0 to 6049
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   name            6050 non-null   str    
 1   address         6050 non-null   str    
 2   score           5978 non-null   float64
 3   nums_comments   5978 non-null   float64
 4   price           4219 non-null   str    
 5   category        6050 non-null   str    
 6   hours           5296 non-null   str    
 7   comments        6047 non-null   str    
 8   comment_scores  6045 non-null   str    
 9   url             6045 non-null   str    
dtypes: float64(2), str(8)
memory usage: 472.8 KB




### 3.4 Các bước tiền xử lý dữ liệu

#### 3.4.1 Chuẩn hóa dữ liệu
Bước chuẩn hóa dữ liệu nhằm đưa các giá trị trong từng cột về định dạng thống nhất, loại bỏ nhiễu và chuyển đổi kiểu dữ liệu phù hợp cho các bước phân tích tiếp theo.

**Cột `Name`**  
- Chuyển về kiểu chuỗi, loại bỏ các khoảng trắng thừa, các ký tự đặc biệt.
- Chuẩn hóa Unicode về dạng NFC (Canonical Decomposition followed by Canonical Composition) để thống nhất cách biểu diễn ký tự tiếng Việt.  

**Cột `Address`**  
- Loại bỏ các dòng có địa chỉ trống hoặc null.  
- Xóa các khoảng trắng, các ký tự dư thừa và chuẩn hóa Unicode NFC tương tự cột `Name`.  
- Xóa hậu tố quốc gia, biến thể tên thành phố và mã bưu chính.

**Cột `Score`**  
Chuyển đổi sang kiểu số thực, thay thế các giá trị không hợp lệ bằng 0, ép kiểu `float32` để tiết kiệm bộ nhớ.

**Cột `Nums_comments`**  
Chuyển đổi sang kiểu số nguyên, thay thế giá trị không hợp lệ bằng 0, ép kiểu `int32`.

**Cột `Price`**  
- Tách cột Price thành hai cột `Price_min` và `Price_max`.   
- Giá trị thiếu hoặc không hợp lệ được gán bằng 0.  
- Xóa cột Price gốc sau khi tách.

**Cột `Category`**  
Loại bỏ khoảng trắng, viết hoa chữ cái đầu và chuyển sang kiểu `category` nhằm tối ưu bộ nhớ và hỗ trợ các thao tác nhóm/lọc.

**Cột `Hours`**  
- Tách cột Hours thành hai cột `Start` và `End`.  
- Nếu giá trị thiếu hoặc không đúng định dạng, gán mặc định là `'-'`.  
- Xóa cột Hours gốc sau khi tách.

**Cột `Comments`**  
- Chuẩn hóa danh sách bình luận về định dạng thống nhất `[{bình luận 1}, {bình luận 2}, ...]`.  
- Loại bỏ ký tự xuống dòng (`\n`), thu gọn khoảng trắng, xóa ký tự thừa.  
- Giá trị trống được gán `'[]'`.

**Cột `Comment_scores`**  
- Chuyển chuỗi điểm phân cách bằng dấu `;` thành danh sách Python (`list[float]`).  
- Giá trị trống được gán danh sách rỗng `[]`.

**Cột `Url`**  
- Rút gọn URL bằng cách loại bỏ phần query string, phần `/data=...` chứa metadata không cần thiết.  

In [40]:
# Header
data.columns = data.columns.str.strip().str.capitalize()

# Name
data['Name'] = (data['Name']
    .astype(str)
    .str.strip()
    .str.replace(r'[*~#]', '', regex=True)
    .apply(lambda x: unicodedata.normalize('NFC', x))
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip(' -,.')
)

# Address
data = data[data['Address'].notna() & (data['Address'].str.strip() != '')].reset_index(drop=True)

def normalize_address(addr):
    if pd.isna(addr) or str(addr).strip() == '':
        return addr
    addr = str(addr).strip()

    addr = unicodedata.normalize('NFC', addr)
    
    addr = re.sub(r',?\s*(Việt Nam|Vietnam)\s*$', '', addr, flags=re.IGNORECASE)
    
    city_patterns = [
        r',?\s*Thành\s+[Pp]hố\s+Hồ\s+Chí\s+Minh',
        r',?\s*TP\.\s*Hồ\s+Chí\s+Minh',
        r',?\s*TP\.\s*HCM',
        r',?\s*TP\.HCM',
        r',?\s*TP\s*HCM',
        r',?\s*Hồ\s+Chí\s+Minh',
    ]
    for pattern in city_patterns:
        addr = re.sub(pattern, '', addr, flags=re.IGNORECASE)
    
    addr = re.sub(r',?\s*\b\d{5,6}\b', '', addr)
    
    addr = re.sub(r'\s*,\s*,\s*', ', ', addr)  
    addr = re.sub(r'\s+', ' ', addr)            
    addr = addr.strip().strip(',').strip()       
    
    return addr

data['Address'] = data['Address'].apply(normalize_address)

# Score
data['Score'] = pd.to_numeric(data['Score'], errors='coerce').fillna(0).astype('float32')

# Nums_comments
data['Nums_comments'] = pd.to_numeric(data['Nums_comments'], errors='coerce').fillna(0).astype('int32')

# Price
def parse_price(val):
    if pd.isna(val) or str(val).strip() == '':
        return (0, 0)
    val_str = str(val).strip()
    # Trường hợp dạng range: "số1-số2"
    if '-' in val_str:
        parts = val_str.split('-')
        try:
            p_min = int(parts[0].strip())
            p_max = int(parts[1].strip())
            return (p_min, p_max)
        except ValueError:
            return (0, 0)
    # Trường hợp đơn giá trị
    try:
        p = int(float(val_str))
        return (p, p)
    except ValueError:
        return (0, 0)

price_parsed = data['Price'].apply(parse_price)
data['Price_min'] = price_parsed.apply(lambda x: x[0]).astype('int32')
data['Price_max'] = price_parsed.apply(lambda x: x[1]).astype('int32')
data.drop(columns=['Price'], inplace=True)

# Category
data['Category'] = data['Category'].str.strip().str.capitalize().astype('category')

# Hours
def parse_hours(val):
    if pd.isna(val) or str(val).strip() == '':
        return ('-', '-')
    parts = str(val).strip().split(' - ')
    if len(parts) == 2:
        return (parts[0].strip(), parts[1].strip())
    return ('-', '-')

hours_parsed = data['Hours'].apply(parse_hours)
data['Start'] = hours_parsed.apply(lambda x: x[0]).astype('str')
data['End'] = hours_parsed.apply(lambda x: x[1]).astype('str')
data.drop(columns=['Hours'], inplace=True)

# Comments
def normalize_comments(val):
    if pd.isna(val) or str(val).strip() in ('', '[]'):
        return '[]'
    
    val_str = str(val).strip()
    
    try:
        items = ast.literal_eval(val_str)
        if isinstance(items, list):
            cleaned = []
            for item in items:
                s = str(item).strip().strip('"').strip("'").strip()
                s = s.replace('\n', ' ').replace('\\n', ' ')
                s = re.sub(r'\s+', ' ', s).strip()
                if s:
                    cleaned.append('{' + s + '}')
            return '[' + ', '.join(cleaned) + ']'
    except (ValueError, SyntaxError):
        pass
    
    if val_str.startswith('[') and val_str.endswith(']'):
        inner = val_str[1:-1]
        parts = re.split(r'\}\s*,\s*\{', inner)
        cleaned = []
        for part in parts:
            s = part.strip().strip('{').strip('}').strip()
            s = s.strip('"').strip("'").strip()
            s = s.replace('\n', ' ').replace('\\n', ' ')
            s = re.sub(r'\s+', ' ', s).strip()
            if s:
                cleaned.append('{' + s + '}')
        return '[' + ', '.join(cleaned) + ']'
    
    s = val_str.replace('\n', ' ').replace('\\n', ' ')
    s = re.sub(r'\s+', ' ', s).strip()
    return '[{' + s + '}]'

data['Comments'] = data['Comments'].apply(normalize_comments)

# Comment_scores
def parse_comment_scores(val):
    if pd.isna(val) or str(val).strip() in ('', '[]'):
        return []
    inner = str(val).strip().strip('[]')
    parts = [p.strip().replace(',', '.') for p in inner.split(';')]
    scores = []
    for p in parts:
        try:
            scores.append(float(p))
        except ValueError:
            continue
    return scores

data['Comment_scores'] = data['Comment_scores'].astype('object').apply(parse_comment_scores)

# Url
def shorten_url(url):
    url = str(url).strip()

    if 'booking.com' in url:
        return url.split('?')[0]

    if 'google.com/maps' in url:
        url = url.split('?')[0]
        url = url.split('/data=')[0]
        return url

    return url

data['Url'] = data['Url'].apply(shorten_url)

# obj_cols = data.select_dtypes(include=['object']).columns.difference(['Comment_scores']).difference(['Comments'])
# data[obj_cols] = data[obj_cols].astype('string')

#### 3.4.2 Trùng lặp dữ liệu

In [41]:
dup_mask = data.duplicated(subset=["Name", "Address"], keep=False)
dup_rows = data[dup_mask].sort_values(["Name", "Address"])
print(f"Dữ liệu trùng nhau: {dup_rows.shape[0]}")

n_removed = data.duplicated(subset=["Name", "Address"], keep="first").sum()
data = data.drop_duplicates(subset=["Name", "Address"], keep="first").reset_index(drop=True)

print(f"Kích thước sau khi xóa trùng lặp: {data.shape}")

Dữ liệu trùng nhau: 2
Kích thước sau khi xóa trùng lặp: (6049, 12)


#### 3.4.3 Tính đúng đắn dữ liệu

In [42]:
print("Score ngoài [0, 5]:", data[(data['Score'] < 0) | (data['Score'] > 5)].shape[0])
print("Nums_comments < 0:", data[data['Nums_comments'] < 0].shape[0])
print("Price_min < 0:", data[data['Price_min'] < 0].shape[0])
print("Price_max < Price_min:", data[data['Price_max'] < data['Price_min']].shape[0])

Score ngoài [0, 5]: 0
Nums_comments < 0: 0
Price_min < 0: 0
Price_max < Price_min: 0


#### 3.4.4 Xử lý cho cột `Address`
Sau bước chuẩn hóa, cột `Address` đã được loại bỏ hậu tố quốc gia, tên thành phố và mã bưu chính. Tuy nhiên, các địa chỉ vẫn còn ở dạng thô với nhiều định dạng khác nhau (thiếu phường/quận, chứa tên tòa nhà, thứ tự không thống nhất,...). Bước này thực hiện gọi **API Gemini** để chuẩn hóa toàn bộ địa chỉ về định dạng thống nhất: **"[số nhà] [tên đường], [phường], [quận]"**, nhằm cung cấp đầy đủ thông tin vị trí và tiện cho việc trích xuất phường/quận ở các tác vụ phân tích sau này.

**Cách thực hiện:**
1. **Trích xuất địa chỉ duy nhất:** Lấy tập các giá trị `Address` không trùng lặp để giảm thiểu số lượt gọi API (thay vì xử lý từng dòng, chỉ xử lý các địa chỉ phân biệt).
2. **Chia batch:** Gom các địa chỉ thành từng lô 50 địa chỉ/request do giới hạn của API key miễn phí.
3. **Gọi API Gemini:** Mỗi batch được gửi kèm prompt yêu cầu LLM chuẩn hóa địa chỉ theo quy tắc:
   - Giữ nguyên số nhà và tên đường.
   - Suy luận phường/quận nếu thiếu nhưng xác định được từ ngữ cảnh.
   - Loại bỏ thông tin phụ (tên tòa nhà, mô tả, số tầng,...).
   - Trả về kết quả dạng JSON có schema cố định (`id`, `address`) với `temperature=0.1` để đảm bảo tính nhất quán.
4. **Xử lý lỗi:** Mỗi batch được thử tối đa 3 lần. Nếu vẫn thất bại, giữ nguyên địa chỉ gốc để không mất dữ liệu.
5. **Ánh xạ ngược:** Xây dựng bảng mapping từ địa chỉ thô sang địa chỉ đã chuẩn hóa, sau đó cập nhật lại toàn bộ cột `Address` trong DataFrame gốc.

**Kết quả:**  
Toàn bộ địa chỉ trong cột `Address` được đưa về định dạng chuẩn, giúp thống nhất cấu trúc dữ liệu và tạo điều kiện thuận lợi cho việc trích xuất thông tin quận, phường phục vụ các bước phân tích và trực quan hóa tiếp theo.

In [ ]:
load_dotenv()

# Khởi tạo Client
client = genai.Client()

class AddressResult(BaseModel):
    id: int
    address: str  

# Hàm gọi API 
def process_address_batch(batch_data, max_retries=3):
    prompt = f"""Bạn là chuyên gia chuẩn hóa địa chỉ tại TP. Hồ Chí Minh, Việt Nam.

Hãy chuẩn hóa các địa chỉ thô sau theo format: "[số nhà] [tên đường], [phường], [quận]"

Quy tắc:
- Giữ lại số nhà và tên đường chính xác.
- Suy luận phường/quận nếu thiếu nhưng có thể xác định được từ ngữ cảnh.
- Nếu không xác định được phường, bỏ qua phần phường: "[số nhà] [đường], [quận]".
- Loại bỏ thông tin phụ không liên quan (tên tòa nhà, mô tả, số tầng...).
- Giữ đúng id tương ứng với từng địa chỉ đầu vào.

Dữ liệu:
{json.dumps(batch_data, ensure_ascii=False)}"""

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema=list[AddressResult],
                    temperature=0.1
                )
            )
            return json.loads(response.text)
        except Exception as e:
            print(f"\nThử lại lần {attempt + 1}: {e}")
            if attempt < max_retries - 1:
                time.sleep(15)  
    return None

# Xử lý các dòng địa chỉ duy nhất -> giảm lượt gọi API
unique_addresses = data['Address'].unique()
print(f"Tổng dòng xử lý: {len(data)} | Địa chỉ duy nhất: {len(unique_addresses)}")
print(f"Số batch cần gọi API: {(len(unique_addresses) - 1) // 50 + 1}")

df_unique = pd.DataFrame({
    'id': range(len(unique_addresses)),
    'raw_address': unique_addresses
})

# Gom các dòng dữ liệu theo batch cho mỗi request (vì dùng API key free bị giới hạn số request)
BATCH_SIZE = 50
all_results = []
failed_batches = []

for i in tqdm(range(0, len(df_unique), BATCH_SIZE), desc="Đang xử lý"):
    chunk = df_unique.iloc[i:i + BATCH_SIZE]
    batch_records = chunk.to_dict('records')

    result = process_address_batch(batch_records)

    if result is not None:
        all_results.extend(result)  
    else:
        failed_batches.append(i)
        # Nếu lỗi thì giữ nguyên địa chỉ ban đầu
        for rec in batch_records:
            all_results.append({"id": rec["id"], "address": rec["raw_address"]})

    time.sleep(5) 

# 6. Map kết quả ngược về DataFrame gốc
result_df = pd.DataFrame(all_results)
merged = df_unique.merge(result_df, on='id', how='left')
address_map = dict(zip(
    merged['raw_address'],
    merged['address'].fillna(merged['raw_address'])
))
data['Address'] = data['Address'].map(address_map).fillna(data['Address'])

Tổng dòng xử lý: 479 | Địa chỉ duy nhất: 477
Số batch cần gọi API: 10


Đang xử lý: 100%|██████████| 10/10 [10:26<00:00, 62.66s/it]


#### 3.4.5 Kết quả đạt đươc

In [43]:
# Thống kê
print("- Kiểu dữ liệu: \n", data.dtypes, "\n")
print("- Dữ liệu thiếu: \n", data.isnull().sum(), "\n")
print("- Thống kê mô tả: \n", data.describe().T, "\n")

# Lưu file kết quả sau khi đã tiền xử lý
data = data[['Name', 'Address', 'Score', 'Nums_comments', 'Price_min', 'Price_max', 'Category', 'Start', 'End', 'Comments', 'Comment_scores', 'Url']]
output_processed = os.path.join(data_dir, 'processed_data.csv')
data.to_csv(output_processed, index=False, encoding='utf-8-sig')

- Kiểu dữ liệu: 
 Name                   str
Address                str
Score              float32
Nums_comments        int32
Category          category
Comments               str
Comment_scores      object
Url                    str
Price_min            int32
Price_max            int32
Start                  str
End                    str
dtype: object 

- Dữ liệu thiếu: 
 Name              0
Address           0
Score             0
Nums_comments     0
Category          0
Comments          0
Comment_scores    0
Url               0
Price_min         0
Price_max         0
Start             0
End               0
dtype: int64 

- Thống kê mô tả: 
                 count           mean           std  min  25%       50%  \
Score          6049.0       3.440899  1.529090e+00  0.0  3.2       4.0   
Nums_comments  6049.0     413.182840  2.192863e+03  0.0  4.0      42.0   
Price_min      6049.0  696407.948587  1.341629e+06  0.0  0.0   50000.0   
Price_max      6049.0  727699.568524  1.333201e+06  

# **Khám phá dữ liệu**

In [49]:
df = pd.read_csv("../data/processed_data.csv")